In [2]:
import pandas as pd
import os
import json

dataset_id = "bigdata-pw/Flickr"
dataset_config = ""
configs_file = "config.json"
compression_methods = ["snappy", "gzip", "brotli", "lz4", "zstd"]
config_dict = json.load(open(configs_file, "r"))[dataset_id]

In [3]:
def get_df(path, max_rows=1_000_000):
    parts = []
    total_rows = 0

    for fname in sorted(os.listdir(path)):
        df_part = pd.read_parquet(os.path.join(path, fname))
        n = len(df_part)

        if total_rows + n >= max_rows:
            # Only take what we still need to reach the cap
            df_part = df_part.iloc[: max_rows - total_rows]
            parts.append(df_part)
            break

        parts.append(df_part)
        total_rows += n

        if total_rows >= max_rows:
            break

    return pd.concat(parts, ignore_index=True)

def compare_dataframes(df1, df2):
    # Align columns by name
    df1_sorted = df1.sort_index(axis=1)
    df2_sorted = df2[df1_sorted.columns]  # reorder df2 to match df1

    # Check shape first
    if df1_sorted.shape != df2_sorted.shape:
        print("DataFrames have different shapes:", df1_sorted.shape, df2_sorted.shape)
        return

    # Create a boolean DataFrame of element-wise comparisons
    diff = df1_sorted != df2_sorted

    if not diff.any().any():
        print("DataFrames are equal")
    else:
        print("Differences found at these locations:")
        differing_cells = diff.stack()[diff.stack()]  # True where differences exist
        for (idx, col) in differing_cells.index:
            print(f"- Row {idx}, Column '{col}': df1 = {df1_sorted.at[idx, col]!r}, df2 = {df2_sorted.at[idx, col]!r}")

def get_size(start_path = '.'):
    # check is it a file or directory
    if os.path.isfile(start_path):
        return os.path.getsize(start_path)
    total = 0
    for dirpath, dirnames, filenames in os.walk(start_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

In [ ]:
df = get_df(config_dict['configs'][0]['path'])
print(df['url_6k'].head())

KeyError: ('url_sq', 'url_q', 'url_t', 'url_s', 'url_n', 'url_w', 'url_m', 'url_z', 'url_c', 'url_l', 'url_h', 'url_k', 'url_3k', 'url_4k', 'url_5k', 'url_6k', 'url_o')

In [5]:
os.makedirs("datasets_compress/bigdata-pw/Flickr/", exist_ok=True)

In [51]:
compress_df = df.copy()
url_cols = ['url_sq', 'url_q', 'url_t', 'url_s', 'url_n', 'url_w', 'url_m', 'url_z', 'url_c', 'url_l']
for url_col in url_cols:
    compress_df[f"{url_col}_null"] = compress_df[url_col].notna()
    # compress_df[f"{url_col}_null"] = compress_df[f"{url_col}_null"].astype('bool')
compress_df['url_temp']= compress_df['url_sq'].apply(lambda x: x.replace("_s", "_{}"))
compress_df = compress_df.drop(columns=url_cols)
print(compress_df.head())

            id          owner  width_sq  height_sq  width_q  height_q  \
0  53908389515  140454096@N02        75         75      150       150   
1  53907222748  142701866@N02        75         75      150       150   
2  53907672754  129787474@N06        75         75      150       150   
3  53905700497  112591365@N05        75         75      150       150   
4  53904629767   86712386@N05        75         75      150       150   

   width_t  height_t  width_s  height_s  ...  url_q_null  url_t_null  \
0      100        67      240       160  ...        True        True   
1      100        67      240       160  ...        True        True   
2      100        75      240       180  ...        True        True   
3      100        79      240       191  ...        True        True   
4       71       100      171       240  ...        True        True   

   url_s_null  url_n_null  url_w_null  url_m_null  url_z_null  url_c_null  \
0        True        True        True        True  

In [52]:
import pyarrow as pa
import pyarrow.parquet as pq

def write_parquet(df, path, PARQUET_COMPRESSION_TYPE="snappy"):
    table = pa.Table.from_pandas(df)
    pq.write_table(table, path, compression=PARQUET_COMPRESSION_TYPE)

if dataset_config == "":
    dataset_config = dataset_id.replace("/", "_").replace("-", "_")
write_parquet(compress_df, f"datasets_compress/{dataset_id}/{dataset_config}.parquet")

In [53]:
compress_df = pd.read_parquet(f"datasets_compress/{dataset_id}/{dataset_config}.parquet")
print(compress_df.info())
print(compress_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 77 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   id              1000000 non-null  string
 1   owner           1000000 non-null  string
 2   width_sq        1000000 non-null  Int32 
 3   height_sq       1000000 non-null  Int32 
 4   width_q         1000000 non-null  Int32 
 5   height_q        1000000 non-null  Int32 
 6   width_t         1000000 non-null  Int32 
 7   height_t        1000000 non-null  Int32 
 8   width_s         1000000 non-null  Int32 
 9   height_s        1000000 non-null  Int32 
 10  width_n         999996 non-null   Int32 
 11  height_n        999996 non-null   Int32 
 12  width_w         999983 non-null   Int32 
 13  height_w        999983 non-null   Int32 
 14  width_m         999956 non-null   Int32 
 15  height_m        999956 non-null   Int32 
 16  width_z         999708 non-null   Int32 
 17  height_z 

In [54]:
size = config_dict['configs'][0]['size_snappy']
compression_size = get_size(f"datasets_compress/{dataset_id}/{dataset_config}.parquet")
print(f"Size of the dataset: {size / (1024):.2f} kB")
print(f"Size of the dataset: {size} B")
print(f"Size of the compression: {compression_size / (1024):.2f} kB")
print(f"Size of the compression: {compression_size} B")
print(f"Compression ratio: {((size-compression_size) / size) * 100:.2f} %")

Size of the dataset: 480020.55 kB
Size of the dataset: 491541045 B
Size of the compression: 279565.98 kB
Size of the compression: 286275566 B
Compression ratio: 41.76 %


In [55]:
url_suffix_dict = {
    'url_sq': '_s',
    'url_q': '_q',
    'url_t': '_t',
    'url_s': '_m',
    'url_n': '_n',
    'url_w': '_w',
    'url_m': '',
    'url_z': '_z',
    'url_c': '_c',
    'url_l': '_b',
    'url_h': '_h',
    'url_k': '_k',
    'url_3k': '_3k',
    'url_4k': '_4k',
    'url_5k': '_5k',
    'url_6k': '_6k',
    'url_o': '_o'
}
for url_col in url_cols:
    compress_df[f"{url_col}"] = compress_df.apply(lambda x: x['url_temp'].replace("_{}", url_suffix_dict[url_col]) if x[f"{url_col}_null"] else None, axis=1)
    compress_df = compress_df.drop(columns=[f"{url_col}_null"])
compress_df = compress_df.drop(columns=['url_temp'])

In [56]:
print(compress_df.head())

            id          owner  width_sq  height_sq  width_q  height_q  \
0  53908389515  140454096@N02        75         75      150       150   
1  53907222748  142701866@N02        75         75      150       150   
2  53907672754  129787474@N06        75         75      150       150   
3  53905700497  112591365@N05        75         75      150       150   
4  53904629767   86712386@N05        75         75      150       150   

   width_t  height_t  width_s  height_s  ...  \
0      100        67      240       160  ...   
1      100        67      240       160  ...   
2      100        75      240       180  ...   
3      100        79      240       191  ...   
4       71       100      171       240  ...   

                                              url_sq  \
0  https://live.staticflickr.com/65535/5390838951...   
1  https://live.staticflickr.com/65535/5390722274...   
2  https://live.staticflickr.com/65535/5390767275...   
3  https://live.staticflickr.com/65535/539057004

In [57]:
compare_dataframes(df, compress_df)

DataFrames are equal
